# Research Replication Roadmap
## *Prompt Repetition Improves Non-Reasoning LLMs* — Leviathan, Kalman & Matias (2025)

> **Source Paper:** arXiv:2512.14982v1 — [https://arxiv.org/abs/2512.14982](https://arxiv.org/abs/2512.14982)

---

## 1. Project Goal

Independently replicate the central finding: that sending a prompt twice — `<QUERY><QUERY>` — improves accuracy of non-reasoning LLMs across standard benchmarks, without increasing output length or latency. You will run experiments across multiple models and benchmarks, perform the same statistical test (McNemar), and write up results as a lab report in the repo's README.

---

## 2. Scope Decisions (Start Here)

Before writing a single line of code, decide on a *realistic* scope. The original paper tested 7 models × 7 benchmarks × multiple configurations. For a first replication:

| Dimension | Original Paper | Recommended Start |
|---|---|---|
| Models | 7 (Gemini, GPT, Claude, Deepseek) | 2–3 (e.g., GPT-4o-mini, Claude Haiku, Gemini Flash Lite) |
| Benchmarks | 7 (incl. custom) | 2–3 (ARC, OpenBookQA, GSM8K) |
| Prompt variants | 5 (baseline, repeat, verbose, ×3, padding) | 2 (baseline + prompt repetition) |
| Reasoning mode | Both (with/without step-by-step) | 1 (without reasoning only) |

Add complexity incrementally once the core pipeline works.

---

## 3. Supplemental Fields of Research to Incorporate

These form the theoretical backbone of your lab report's literature review section.

### 3a. Core Mechanistic Background
- **Causal (autoregressive) language modeling** — Why token order matters; the unidirectional attention constraint. Read the original GPT-2 paper (Radford et al., 2019) for grounding.
- **Attention mechanisms & transformers** — "Attention is All You Need" (Vaswani et al., 2017). The key insight: in standard causal LMs, an early context token can't *attend back* to a later question token. Prompt repetition removes this asymmetry.
- **Lost in the middle / position bias** — Liu et al. (2023), *"Lost in the Middle: How Language Models Use Long Contexts"*. Supports why middle-of-prompt tokens can be underweighted.

### 3b. Prompting Technique Literature
- **Chain of Thought (CoT)** — Wei et al. (2023). Contrast: CoT increases output length; prompt repetition does not.
- **"Think step by step"** — Kojima et al. (2023). Same contrast — compute cost difference.
- **Re-reading (RE2)** — Xu et al. (2024), *"Re-reading Improves Reasoning in LLMs"*. Closely related: asks the model to re-read *during generation*; prompt repetition does it in the *prefill* stage.
- **Prefix LM vs. Causal LM** — Raffel et al. (2023), T5 paper. Motivates why bidirectional context helps.

### 3c. Statistical Methods
- **McNemar's Test** — The paper's primary significance test. Study its mechanics: it tests correlated binary outcomes (did model A get it right when model B got it wrong, and vice versa). Key references: McNemar (1947); any introductory biostatistics text covers it.
- **Effect size reporting** — Consider reporting odds ratios alongside p-values for richer analysis.

### 3d. Efficiency / Inference Background
- **Speculative decoding** — Leviathan et al. (2022) (same first author). The paper draws a parallel: prompt repetition is efficient because it only adds to the *parallelizable prefill* stage, not the sequential decode stage. Understanding this distinction strengthens your analysis section.
- **KV-cache mechanics** — Read Hugging Face's blog on KV caching. Explains why prefill is cheap relative to decode.

---

## 4. Tools & Stack

### 4a. Core Dependencies

```
# requirements.txt

# LLM API clients
anthropic>=0.49.0
openai>=1.75.0
google-generativeai>=0.8.0

# Data & benchmarks
datasets>=3.0.0          # HuggingFace datasets (ARC, OpenBookQA, GSM8K, MMLU-Pro)
huggingface-hub>=0.24.0

# Numerical & statistical
numpy>=1.26.0
pandas>=2.2.0
scipy>=1.13.0            # McNemar test via scipy.stats.mcnemar

# Experiment orchestration
langchain>=0.3.0         # Optional but useful for prompt templates & model abstraction
langchain-openai
langchain-anthropic
langchain-google-genai

# Timing & logging
tqdm>=4.66.0
python-dotenv>=1.0.0     # API key management

# Visualization
matplotlib>=3.9.0
seaborn>=0.13.0

# Testing
pytest>=8.0.0
pytest-asyncio>=0.23.0

# Reporting
jupyterlab>=4.2.0        # For exploratory analysis notebooks
```

### 4b. Notes on Tool Choices

**Why LangChain?** The paper tests multiple providers (OpenAI, Anthropic, Google). LangChain's `BaseChatModel` abstraction lets you write one prompt-runner that works across all of them. This is a genuine use case, not over-engineering.

**Why `scipy.stats.mcnemar`?** The paper uses the McNemar test as its significance criterion. `scipy` has a direct implementation. You should also implement it manually once to understand what it's computing (a 2×2 contingency table of per-question right/wrong outcomes).

**Why `datasets`?** ARC, OpenBookQA, and GSM8K are all natively available on HuggingFace Hub and load in one line. This keeps your data pipeline clean and reproducible.

---

## 5. Data Sources

| Benchmark | HuggingFace Handle | Split to Use | Notes |
|---|---|---|---|
| ARC Challenge | `allenai/ai2_arc`, config `ARC-Challenge` | `test` | Multiple-choice; test both question-first and options-first ordering |
| OpenBookQA | `allenai/openbookqa`, config `main` | `test` | Multiple-choice; same ordering configs |
| GSM8K | `openai/gsm8k`, config `main` | `test` | Math word problems; no ordering variant |
| MMLU-Pro | `TIGER-Lab/MMLU-Pro` | `test` | Harder; add in phase 2 |
| MATH | `lighteval/MATH` | `test` | Hardest; add in phase 3 |

**Custom benchmarks (NameIndex, MiddleMatch):** Generate these yourself. The paper describes the construction in Appendix A.3. Write a `data/generators/` module that produces them programmatically. This is actually a good confidence-builder — if your custom task results match the paper's dramatic gains, you know your pipeline is working.

---

## 6. Repo Structure

```
prompt-repetition-replication/
│
├── README.md                        # Lab report (see Section 9 below)
├── requirements.txt
├── .env.example                     # API key template (never commit .env)
├── pyproject.toml                   # Optional: packaging config
│
├── agents/
│   ├── __init__.py
│   ├── base_agent.py                # Abstract LLM agent class
│   ├── openai_agent.py              # GPT wrapper
│   ├── anthropic_agent.py           # Claude wrapper
│   ├── google_agent.py              # Gemini wrapper
│   └── prompt_builder.py            # Constructs baseline, repeat, verbose, ×3, padding variants
│
├── data/
│   ├── loaders/
│   │   ├── arc_loader.py
│   │   ├── obqa_loader.py
│   │   └── gsm8k_loader.py
│   ├── generators/
│   │   ├── name_index.py            # Custom NameIndex task generator
│   │   └── middle_match.py          # Custom MiddleMatch task generator
│   └── cache/                       # gitignored; stores downloaded datasets locally
│
├── experiments/
│   ├── config.py                    # Experiment configuration (models, benchmarks, n_samples)
│   ├── runner.py                    # Main experiment loop
│   ├── evaluator.py                 # Parses model responses, checks correctness
│   └── results/                     # gitignored; raw JSON result files per run
│
├── analysis/
│   ├── mcnemar_test.py              # McNemar test implementation & wrapper
│   ├── statistics.py                # Effect sizes, confidence intervals
│   ├── visualize.py                 # Bar charts replicating Figure 1 style
│   └── notebooks/
│       └── explore_results.ipynb    # EDA on saved results
│
└── tests/
    ├── test_prompt_builder.py       # Unit tests for prompt variants
    ├── test_evaluator.py            # Unit tests for answer parsing
    ├── test_mcnemar.py              # Verify McNemar against known values
    └── test_loaders.py              # Smoke tests for data loaders
```

---

## 7. Implementation Roadmap (Phased)

### Phase 1 — Scaffold & Sanity Check (Week 1)
- [ ] Set up repo, virtual environment, `.env` with API keys
- [ ] Implement `prompt_builder.py` with all 5 variants (start with baseline + repeat)
- [ ] Write `test_prompt_builder.py` — verify that `<QUERY><QUERY>` is exactly double the baseline, no whitespace bugs
- [ ] Implement one data loader (`arc_loader.py`), write its test
- [ ] Implement one agent (`openai_agent.py`), manually test with 5 ARC questions in a Jupyter notebook
- [ ] Implement `evaluator.py` for multiple-choice answer parsing (regex on "The answer is X")

### Phase 2 — Core Experiment Pipeline (Week 2)
- [ ] Implement `runner.py`: loops over (models × benchmarks × prompt_variants × n_samples)
- [ ] Save results as JSON per run: `{model, benchmark, variant, question_id, correct: bool, response_tokens: int, latency_ms: float}`
- [ ] Implement `mcnemar_test.py`: build contingency table from paired results, run `scipy.stats.mcnemar`
- [ ] Run a small pilot: 50 questions, 1 model, baseline vs. repeat, confirm you see directional improvement
- [ ] Add remaining data loaders and agents

### Phase 3 — Full Replication Run (Week 3)
- [ ] Run full experiment: your chosen subset (2–3 models × 3 benchmarks × 2 variants)
- [ ] Generate visualizations replicating Figure 1's grouped bar chart format
- [ ] Implement custom task generators and run NameIndex/MiddleMatch — these show the most dramatic gains
- [ ] Check latency and response length results (should be neutral between baseline and repeat)

### Phase 4 — Analysis & Write-up (Week 4)
- [ ] Tabulate McNemar test results (wins/losses/ties)
- [ ] Compare your win rate to paper's 47/70
- [ ] Document any divergences and hypothesize why (model version differences, sampling, prompt formatting)
- [ ] Write the README lab report (see Section 9)
- [ ] Write `analysis/notebooks/explore_results.ipynb` as a reproducible analysis walkthrough

---

## 8. Key Implementation Details & Gotchas

### Prompt Formatting
The paper is specific: the input is transformed from `" <QUERY> "` to `" <QUERY><QUERY> "`. Match this exactly. There's no separator between repetitions in the baseline variant. The verbose variant adds `"Let me repeat that:\n"` between them.

### Answer Parsing
Multiple-choice benchmarks need careful parsing. The paper instructs models to reply in the format `"The answer is <ANSWER>."` — build this instruction into every prompt. Your `evaluator.py` should:
1. Try to parse the exact format first
2. Fall back to finding the first standalone A/B/C/D letter in the response
3. Flag ambiguous responses for manual review (log them)

### Cost Management
Running experiments across 3 models × 3 benchmarks × 500 questions × 2 variants = ~9,000 API calls. Estimate costs before running:
- GPT-4o-mini: ~$0.60/1M input tokens (very cheap)
- Claude Haiku: ~$0.80/1M input tokens (very cheap)
- Gemini Flash Lite: free tier available

Use `n_samples` config to run 100-question pilots before full runs. Cache results to disk so you never re-run completed experiments.

### Statistical Validity
McNemar test requires *paired* results — the same questions answered by both baseline and repeated-prompt conditions. Ensure your `runner.py` evaluates the same question IDs in both conditions. Shuffle order to avoid position effects, but track IDs.

### Latency Measurement
Measure end-to-end latency (`time.perf_counter()` around the API call). The paper notes network variability — run all providers in round-robin fashion to be fair. Report median, not mean (more robust to outliers).

---

## 9. README as Lab Report — Outline

Your README should read as a first-person research article. Suggested structure:

```
# Replication Study: Prompt Repetition Improves Non-Reasoning LLMs

## Abstract
## 1. Introduction & Motivation
## 2. Source Publication
   - Full citation, link, authors
## 3. Theoretical Background
   - Causal LMs and the attention asymmetry problem
   - Related prompting techniques and how this differs
## 4. Methods
   - Models tested
   - Benchmarks used
   - Prompt variants implemented
   - Statistical test (McNemar) — explain it in plain language
   - Experimental setup (API versions, dates run, n_samples)
## 5. Results
   - Win/loss/tie table vs. paper's 47/70
   - Reproduced Figure 1 (your version)
   - Latency and output length findings
   - Custom task results
## 6. Discussion
   - Where you replicated successfully
   - Divergences and possible explanations
   - Limitations (cost constraints, model version differences)
## 7. Conclusion
## 8. References
   - Source paper + all supplemental literature cited
## Appendix: Reproduction Instructions
   - How to run the experiments from scratch
```

---

## 10. Stretch Goals (After Core Replication)

Once the base replication is solid, these extensions add original contribution:

1. **Test prompt repetition ×3 vs. ×2** — The paper notes ×3 outperforms ×2 on custom tasks but doesn't fully explore why.
2. **Vary prompt structure** — Does putting the second repetition in the system prompt vs. user turn change anything?
3. **Measure token probability distributions** — Using logprobs (OpenAI API supports this), compare confidence distributions between baseline and repeated prompts.
4. **Test on non-English prompts** — The paper doesn't explore multilingual settings.
5. **Fine-tuning hypothesis** — The paper suggests fine-tuning on repeated prompts as future work. You could test this with a small LoRA fine-tune on an open-source model.

---

*Roadmap version 1.0 — March 2026*
